# 04-LangGraph Foundations (State & Nodes)

In Lesson 03, we perfected our agent's cognitive structure using Pydantic Coercion. We can mathematically guarantee that any data generated by an LLM strictly conforms to our required topologies.

But where does this structured data go? In a single-turn application, we simply return it to the user. But in an autonomous, multi-step Agentic Workflow, this data must be preserved, mutated, and passed forward into the next computational step.

We must graduate from simple sequential chains into the mathematics of **Graph Theory and State Machines**. In 2024, the enterprise standard for this is **LangGraph**.

Let's set up our environment to engineer the physics of the `StateGraph`.

In [ ]:
from typing import TypedDict, Annotated, Sequence
import operator
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Systems Architecture & Graph State Environment Ready.")

# 1. The Physics of the State Machine ($S_t$)

At the core of LangGraph is a single, unified mathematical payload called the **State** ($S$).

In standard Python programming, you pass variables individually between functions: `y = func(x, z)`.
In a LangGraph State Machine, **all functions accept the exact same object ($S$) and return a delta ($\Delta S$) to update it.**

Let $S_t$ be the state of our agent at time $t$. Let $N$ be a Node (a Python function).
The computation resolves as:


$$S_{t+1} = S_t \oplus N(S_t)$$

The $\oplus$ operator is critical. When a node returns data, LangGraph does not blindly overwrite the old state. It uses **Reducers** to merge the data.

### The `TypedDict` and the Reducer Matrix

We define the structure of $S$ using a Python `TypedDict`. For every variable in the State, we define its type and its **Reducer**.

* **Overwrite (Default)**: If Node A returns `{"temperature": 0.8}`, the state's temperature is completely replaced.
* **Append (`Annotated[list, operator.add]`)**: If Node A returns `{"messages": ["Hello"]}`, LangGraph appends "Hello" to the existing list of messages, preserving the entire conversational history.

# 2. Architecting the Nodes and Edges

A LangGraph is built using three primitives:

1. **Nodes ($V$)**: Pure Python functions that contain the computational logic (e.g., calling an LLM, querying a database). A node accepts $S_t$ and returns $\Delta S$.
2. **Edges ($E$)**: The directional routing logic. Standard edges connect Node A strictly to Node B. (We will explore *Conditional Edges* in Lesson 05).
3. **The Graph ($G$)**: The compilation of Nodes and Edges into an executable LangChain `Runnable`.

# 3. Architecting a Foundational StateGraph

Let's build a functional, multi-node LangGraph from scratch. We will engineer an AI Research pipeline with two distinct nodes: a **Researcher Node** (gathers facts) and a **Writer Node** (synthesizes the facts). We will watch the State mutate at every hop.

*(Note: We will simulate the LLM calls using string manipulation to isolate and prove the graph physics).*

In [ ]:
# --- 🕸️ The LangGraph State Machine Engine ---

# In production, you would import langgraph.graph:
# from langgraph.graph import StateGraph, START, END

# 1. Define the Mathematical State (The Payload)
class AgentState(TypedDict):
    # 'query' overwrites. It holds the user's initial prompt.
    query: str
    
    # 'research_facts' uses a Reducer (operator.add). 
    # Any node returning facts will APPEND them to this list, not overwrite it!
    research_facts: Annotated[list[str], operator.add]
    
    # 'final_draft' overwrites.
    final_draft: str

# 2. Define the Computational Nodes
def researcher_node(state: AgentState) -> dict:
    """Reads the query, simulates a web search, and appends facts to the State."""
    print("   🔍 [NODE: Researcher] Executing fact retrieval...")
    query = state["query"]
    
    # Simulated Tool Execution
    new_facts = [
        f"Fact 1: '{query}' was heavily searched in 2024.",
        f"Fact 2: Enterprise adoption of '{query}' grew by 40%."
    ]
    
    # We return ONLY the delta. LangGraph will auto-append this to 'research_facts'
    return {"research_facts": new_facts}

def writer_node(state: AgentState) -> dict:
    """Reads the accumulated facts and overwrites the final_draft state."""
    print("   ✍️ [NODE: Writer] Synthesizing facts into a final draft...")
    facts = state.get("research_facts", [])
    
    # Simulated LLM Synthesis
    draft = f"Executive Summary:\n"
    for fact in facts:
        draft += f"- {fact}\n"
    
    return {"final_draft": draft}

# 3. Simulate Graph Compilation and Execution
# In a real LangGraph environment, you do this:
# workflow = StateGraph(AgentState)
# workflow.add_node("Researcher", researcher_node)
# workflow.add_node("Writer", writer_node)
# workflow.add_edge(START, "Researcher")
# workflow.add_edge("Researcher", "Writer")
# workflow.add_edge("Writer", END)
# app = workflow.compile()

print("--- 🚀 Initializing StateGraph Execution ---")

# The Initial State (S_0)
S_0: AgentState = {"query": "Quantum Computing", "research_facts": [], "final_draft": ""}
print(f"S_0 (Start) : {S_0}\n")

# Transition 1: START -> Researcher
delta_1 = researcher_node(S_0)
# The framework mathematically reduces the state: S_1 = S_0 + delta_1
S_1 = {
    "query": S_0["query"], 
    "research_facts": S_0["research_facts"] + delta_1["research_facts"], # Appended!
    "final_draft": S_0["final_draft"]
}
print(f"S_1 (Update): {S_1}\n")

# Transition 2: Researcher -> Writer
delta_2 = writer_node(S_1)
# S_2 = S_1 + delta_2
S_2 = {
    "query": S_1["query"], 
    "research_facts": S_1["research_facts"], 
    "final_draft": delta_2["final_draft"] # Overwritten!
}
print(f"S_2 (Final) : \n{S_2['final_draft']}")

print("--- 💡 Engineering Insight ---")
print("Notice how the Nodes communicate! The Writer Node has no idea who the Researcher Node is. It doesn't accept arguments directly from the Researcher. Both nodes are completely decoupled functions that only interact by reading and mutating the shared 'AgentState' clipboard.")

# 4. Visualizing Graph Topology

To mathematically prove how state flows through an orchestrated network, let's visualize the Directed Acyclic Graph (DAG) we just built, tracking the precise moment the Reducers act on the State schema.

In [ ]:
# Visualizing the StateGraph Topology
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("LangGraph Topology: State Mutation over Edges", fontsize=16, fontweight='bold')

ax.axis('off')

# 1. Draw Nodes (Vertices)
nodes = {
    "__START__": (1, 3, "black", "white"),
    "Researcher": (5, 3, "#3498db", "white"),
    "Writer": (9, 3, "#9b59b6", "white"),
    "__END__": (13, 3, "black", "white")
}

for name, (x, y, color, text_color) in nodes.items():
    if name in ["__START__", "__END__"]:
        circle = patches.Circle((x, y), 0.6, color=color, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, name.strip("_"), color=text_color, fontweight='bold', ha='center', va='center', fontsize=10, zorder=4)
    else:
        rect = patches.FancyBboxPatch((x-1, y-0.6), 2, 1.2, boxstyle="round,pad=0.1", 
                                     linewidth=2, edgecolor='black', facecolor=color, zorder=3)
        ax.add_patch(rect)
        ax.text(x, y, name, color=text_color, fontweight='bold', ha='center', va='center', fontsize=12, zorder=4)

# 2. Draw Edges and State Updates
# START -> Researcher
ax.annotate("", xy=(4, 3), xytext=(1.6, 3), arrowprops=dict(arrowstyle="->", lw=3, color="gray"))
ax.text(2.8, 3.2, "$S_0$\nInitialize", ha='center', fontsize=10, fontweight='bold', color="black")

# Researcher -> Writer
ax.annotate("", xy=(8, 3), xytext=(6, 3), arrowprops=dict(arrowstyle="->", lw=3, color="gray"))
ax.text(7, 3.2, "$S_1 = S_0 \\oplus \\Delta S$\n(Append Facts)", ha='center', fontsize=10, fontweight='bold', color="blue")

# Writer -> END
ax.annotate("", xy=(12.4, 3), xytext=(10, 3), arrowprops=dict(arrowstyle="->", lw=3, color="gray"))
ax.text(11.2, 3.2, "$S_2 = S_1 \\oplus \\Delta S$\n(Overwrite Draft)", ha='center', fontsize=10, fontweight='bold', color="purple")

# 3. Draw the Global State Object (The Clipboard)
state_box = patches.Rectangle((4.5, 0), 5, 1.5, fill=True, color='#f1c40f', alpha=0.2, linewidth=2, edgecolor='black')
ax.add_patch(state_box)
ax.text(7, 0.75, "Global AgentState (TypedDict)\n{ query: str, facts: list, draft: str }", 
        ha='center', va='center', color='black', fontweight='bold', fontsize=11)

# Draw lines connecting nodes to the Global State
ax.plot([5, 5], [2.4, 1.5], color='gray', linestyle='--', linewidth=2)
ax.plot([9, 9], [2.4, 1.5], color='gray', linestyle='--', linewidth=2)

plt.xlim(0, 14)
plt.ylim(-0.5, 4.5)
plt.tight_layout()
plt.show()

print("\n--- 💡 Engineering Insight ---")
print("Look at the Yellow Box. This is the magic of LangGraph. In standard Python scripts, moving data through 10 different functions requires massive, tangled return statements. Here, the State object acts as a Global Clipboard. Nodes just grab the clipboard, read what they need, write their updates, and pass it down the line.")

## Real-World Use Case or Analogy:

Think of a LangGraph State Machine like a **Hospital Patient Chart**:

* **The State ($S$)**: The physical clipboard attached to the patient's bed (The `AgentState`). It has strict sections: Patient Name (Overwrite), Symptoms (Append), Final Diagnosis (Overwrite).
* **The Nodes ($V$)**: The hospital staff.
* Node 1 is the **Triage Nurse**. They read the Name, check the temperature, and *append* a symptom to the chart. They hand the clipboard to the doctor.
* Node 2 is the **Doctor**. They don't need to talk to the nurse. They just read the chart (State), realize there are 3 symptoms appended, formulate a diagnosis, and *overwrite* the blank Diagnosis line.


* **The Edges ($E$)**: The physical path the clipboard takes down the hallway.

Because the staff (Nodes) communicate entirely through the standardized chart (State), you can infinitely scale the hospital. You can add an X-Ray Tech Node, a Billing Node, and a Pharmacy Node. They all just read and mutate the exact same clipboard as it travels along the edges.